# Clase 146 — CLIP / SigLIP embeddings multimodales

CLIP entrena texto + imagen para mapear ambos al *mismo espacio* (cosine similarity). Habilita zero-shot classification y image search.

In [ ]:
USE_ST = False
try:
    from sentence_transformers import SentenceTransformer
    USE_ST = True
    print('sentence_transformers disponible')
except Exception as e:
    print('ST no disponible. Fallback embeddings random tagged. Motivo:', type(e).__name__)

import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

## 1. Textos + 'imágenes' sintéticas

In [ ]:
texts = ['a cat sitting on a chair', 'a dog running in a park', 'a red apple on a table', 'a blue car on a road', 'a sunset over the ocean']
image_tags = ['cat_indoor', 'dog_outdoor', 'apple_kitchen', 'car_street', 'sunset_beach']
print('pairs:', list(zip(texts, image_tags)))

## 2. Encode (CLIP real o fallback)

In [ ]:
if USE_ST:
    try:
        model = SentenceTransformer('clip-ViT-B-32')
        text_emb = model.encode(texts, normalize_embeddings=True)
        # Para CPU/sin imágenes reales, simulamos imagen como mismo modelo sobre captions
        img_emb = model.encode(texts, normalize_embeddings=True)   # idealmente sería Image
        print('CLIP embeddings:', text_emb.shape)
    except Exception:
        USE_ST = False
if not USE_ST:
    # fallback: a cada par (text, image) le damos un vector cercano (mismo seed por par)
    def fake_embed(tag):
        rng = np.random.default_rng(hash(tag) % (2**32))
        return rng.normal(0, 1, 128)
    text_emb = np.array([fake_embed(t.split()[1]) + np.random.normal(0, 0.1, 128) for t in texts])
    img_emb = np.array([fake_embed(t.split('_')[0]) + np.random.normal(0, 0.1, 128) for t in image_tags])
    text_emb /= np.linalg.norm(text_emb, axis=1, keepdims=True)
    img_emb /= np.linalg.norm(img_emb, axis=1, keepdims=True)
    print('fallback embeddings:', text_emb.shape)

## 3. Matriz de similitud texto-imagen

In [ ]:
sim = text_emb @ img_emb.T          # (n_text, n_img)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(image_tags))); ax.set_xticklabels(image_tags, rotation=45, ha='right')
ax.set_yticks(range(len(texts))); ax.set_yticklabels([t[:25] for t in texts])
plt.colorbar(im, ax=ax, label='cosine sim')
for i in range(len(texts)):
    for j in range(len(image_tags)):
        ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center', fontsize=8)
ax.set_title('text–image similarity'); plt.tight_layout(); plt.show()

## 4. Zero-shot classification

Dado una imagen `i`, su clase = `argmax_j sim(text_j, img_i)`. Sin entrenar.

In [ ]:
for i, tag in enumerate(image_tags):
    pred = sim[:, i].argmax()
    correct = '✓' if pred == i else '✗'
    print(f'{correct} image[{tag}] → predicted text: "{texts[pred]}" (sim={sim[pred, i]:.3f})')
acc = (sim.argmax(0) == np.arange(len(texts))).mean()
print(f'\nzero-shot accuracy: {acc:.2%}')

## 5. Image search: dado query text, encuentra la mejor imagen

In [ ]:
query = 'an apple'
if USE_ST:
    q_emb = model.encode([query], normalize_embeddings=True)[0]
else:
    q_emb = fake_embed('apple') / np.linalg.norm(fake_embed('apple'))
scores = img_emb @ q_emb
for i in np.argsort(-scores):
    print(f'  {image_tags[i]:18s}  sim={scores[i]:+.3f}')

## 6. SigLIP vs CLIP

| Aspecto | CLIP | SigLIP |
|---------|------|--------|
| Loss | softmax contrastive (cada batch) | sigmoid pairwise (independiente) |
| Batch size | requiere batches grandes (~32k) | escala con batches chicos |
| Negativos | implícitos en el batch | explícitos por par |
| Robustez | sensible a label noise | más estable |

```python
# CLIP loss (simplificado)
logits = (T @ I.T) * temp
loss = (cross_entropy(logits, eye) + cross_entropy(logits.T, eye)) / 2

# SigLIP loss
logits = (T @ I.T) * temp + bias
labels = 2 * eye - 1                    # +1 pares match, -1 no-match
loss = -log_sigmoid(labels * logits).mean()
```

## Conclusiones

- CLIP (OpenAI 2021): dual-encoder + contrastive loss → embeddings alineados texto-imagen.
- Habilita zero-shot, retrieval, RAG multimodal sin fine-tuning.
- SigLIP (Google 2023) reemplaza softmax con sigmoid → mejor con batches chicos.
- Variantes: CLIP-ViT-L/14, EVA-CLIP, OpenCLIP, SigLIP-2.
- En prod: index con FAISS sobre img embeddings; query encode + ANN.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README de esta clase. El código que usa librerías pesadas (`transformers` / `torch` / `keras` / `diffusers`) es la **API real** de la industria y se valida por sintaxis (los modelos requieren GPU/descarga). Los **núcleos numéricos** están en numpy puro, son **ejecutables** y se autoverifican con `assert`.

In [ ]:
# Deteccion del stack real (CLIP via transformers). Si no esta, se valida el
# nucleo de coseno en numpy sobre los embeddings del notebook.
try:
    import torch  # noqa: F401
    from transformers import CLIPModel  # noqa: F401
    _HF = True
except Exception:
    _HF = False
print('CLIP/transformers disponible:', _HF)

### Ejercicio 1 — CLIP setup + cosine similarity (núcleo numpy ejecutable)

In [ ]:
# --- API real: cargar processor + model, embeber imagen y texto ---
if _HF:
    from transformers import CLIPModel, CLIPProcessor
    from PIL import Image
    m = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
    proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
    inp = proc(text=['a photo of a cat'], images=Image.open('cat.jpg'),
               return_tensors='pt', padding=True)
    out = m(**inp)
    te = out.text_embeds / out.text_embeds.norm(dim=-1, keepdim=True)
    ie = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
    print('cosine texto-imagen:', float(te @ ie.T))
else:
    print('CLIP real requiere transformers+torch. Verifico el coseno en numpy abajo.')

# --- Nucleo numpy ejecutable: coseno sobre los embeddings del notebook ---
import numpy as np
def cosine(u, v):
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-9))

assert abs(cosine(text_emb[0], text_emb[0]) - 1.0) < 1e-6      # consigo mismo = 1
S = text_emb @ img_emb.T                                       # (n_text, n_img)
diag = np.trace(S) / len(S)
off = (S.sum() - np.trace(S)) / (S.size - len(S))
print(f'sim media diagonal (pares correctos) = {diag:.3f}')
print(f'sim media fuera de diagonal          = {off:.3f}')
assert diag > off      # los pares texto-imagen correctos son mas similares
print('OK: los pares alineados dominan la matriz de similitud')

### Ejercicio 2 — Zero-shot classification (argmax de similitud)

In [ ]:
# API real: comparar una imagen contra plantillas de texto.
if _HF:
    labels = ['a photo of a cat', 'a photo of a dog']
    # inp = proc(text=labels, images=img, return_tensors='pt', padding=True)
    # probs = m(**inp).logits_per_image.softmax(-1)   # prob por label
    print("logits_per_image.softmax(-1) -> argmax = clase (sin entrenar).")

# Nucleo numpy: para cada imagen, el texto mas similar.
preds = S.argmax(axis=0)                       # texto ganador por columna (imagen)
acc = float((preds == np.arange(len(texts))).mean())
print(f'zero-shot accuracy (fallback) = {acc:.0%}')
assert acc >= 0.6

### Ejercicio 3 — Image search: query de texto → top-k imágenes

In [ ]:
# Nucleo numpy: query 'a sunset over the ocean' -> ranking de imagenes.
q = text_emb[4]                                # embedding del texto 'sunset'
sims = np.array([cosine(q, img_emb[j]) for j in range(len(image_tags))])
order = np.argsort(-sims)
print('ranking:', [f'{image_tags[i]} ({sims[i]:+.2f})' for i in order])
assert order[0] == 4      # la imagen del atardecer rankea primero

if _HF:
    print("En prod: encode del corpus de imagenes -> FAISS; encode del query -> ANN top-k.")

### Ejercicio 4 — SigLIP: misma tarea, loss sigmoide

In [ ]:
# API real: SigLIP (Google 2023) usa sigmoid pairwise en vez de softmax.
if _HF:
    from transformers import AutoModel, AutoProcessor
    sm = AutoModel.from_pretrained('google/siglip-base-patch16-224')
    sp = AutoProcessor.from_pretrained('google/siglip-base-patch16-224')
    # inp = sp(text=texts, images=imgs, return_tensors='pt', padding='max_length')
    # logits = sm(**inp).logits_per_image        # ya escalados, aplicar sigmoid
    print('SigLIP: logits_per_image -> sigmoid (cada par se juzga independiente).')
else:
    print('SigLIP loss: -log_sigmoid(labels * (T@I.T * temp + bias)),'
          ' labels=+1 par match / -1 no-match. Escala con batches chicos.')

### Ejercicio 5 — Fine-tune ligero de CLIP con LoRA

In [ ]:
# API real: adaptar CLIP a un dominio con PEFT/LoRA sobre las proyecciones.
if _HF:
    from peft import LoraConfig, get_peft_model
    cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                     target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj'])
    # clip_lora = get_peft_model(m, cfg)
    # entrenar con InfoNCE sobre pares (imagen, caption) del dominio custom.
    print('CLIP + LoRA(r=8) sobre q/k/v/out_proj; loss contrastiva InfoNCE.')
else:
    print('Fine-tune ligero: congelar CLIP, entrenar adapters LoRA con pares'
          ' (imagen, caption) del dominio; ~0.5% de parametros entrenables.')